# SYNAPSE generator-learning and numerical audit

Select an **A100** runtime. This reads the completed pilot and uses **A/M only**. It does not rerun training/reporting or evaluate D/test. The old Drive folder stays unchanged.

[Design and limitations](https://github.com/akashm776/SYNAPSE/blob/main/docs/GENERATOR_AUDIT.md). Default: one seed, two teacher states, two A/M pairs, repeatability/precision comparisons and 20-step fixed-set fitting. The 30-minute budget is checked between units, not a hard timeout. GPU feasibility of this new audit is not yet established.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
REPO = Path('/content/SYNAPSE')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/akashm776/SYNAPSE.git', str(REPO)], check=True)
CODE_REF = ''  # On resume, use the exact revision printed by the first audit setup.
if CODE_REF:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', CODE_REF], check=True)
elif not (REPO / 'synapse/audit.py').exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
print('Record audit revision:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.9.1', 'torchvision==0.24.1', 'torchaudio==2.9.1', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)+'[llm,test]', 'numpy==2.3.5', 'accelerate==1.2.1'], check=True)
subprocess.run([sys.executable, '-c', "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name()); assert 'A100' in torch.cuda.get_device_name()"], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SOURCE = Path('/content/drive/MyDrive/SynLess/runs/llm_qwen_v2')
OUTPUT = Path('/content/drive/MyDrive/SYNAPSE/audits/qwen_generator_v1')
CONFIG = REPO / 'configs/audit_qwen_a100.json'
assert (SOURCE / 'data.json').exists(), 'Use the complete original Drive run, not the public metric archive'
print('Read-only source:', SOURCE)
print('Separate audit output:', OUTPUT)
print(CONFIG.read_text())


## Run or resume the audit

Rerun this cell after a budget pause. Completed groups/fit steps are reused. For a runtime restart, first restore the same CODE_REF and environment. Do not run two sessions into the same output folder.


In [ ]:
subprocess.run([sys.executable, '-m', 'synapse.audit', '--source', str(SOURCE), '--output', str(OUTPUT), '--config', str(CONFIG)], cwd=REPO, check=True)
status = json.loads((OUTPUT / 'audit_manifest.json').read_text())['status']
print('Audit status:', status)
if status == 'complete':
    print((OUTPUT / 'AUDIT_REPORT.md').read_text())
else:
    print('Rerun this cell to resume. This is not a completed diagnostic yet.')


## Share results

Share `audit_manifest.json`, `audit_summary.json`, and `AUDIT_REPORT.md`. Keep `state_*/fit_*.pt` for resume. These are optimization/numerical diagnostics, not evidence of held-out fine-tuning gains. No new original-pilot run or official-test report is needed.
